# Practical P16: First Real AI API Call from Python
**Learning Outcome**: Call an AI API using raw HTTP requests — understanding the mechanism under SDKs.

## Part 1: Local Mock LLM API Setup for testing
To simulate calling a real LLM API (without incurring actual API costs or requiring active keys), we will spin up a lightweight mock LLM HTTP server in a background thread inside this notebook.


In [ ]:
import http.server
import socketserver
import threading
import json
import time

class MockLLMRequestHandler(http.server.BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        pass # Suppress server logging stdout
        
    def do_POST(self):
        content_length = int(self.headers.get('Content-Length', 0))
        body = self.rfile.read(content_length)
        
        # Check Authorization Header
        auth = self.headers.get('Authorization')
        if not auth or not auth.startswith('Bearer '):
            self.send_response(401)
            self.end_headers()
            self.wfile.write(b'{"error": "Unauthorized: Missing or invalid token"}')
            return
            
        payload = json.loads(body.decode('utf-8'))
        model = payload.get('model', 'unknown-model')
        messages = payload.get('messages', [])
        prompt = messages[-1]['content'] if messages else ''
        
        # Generate response payload
        resp = {
            'id': 'chatcmpl-mock-11111',
            'object': 'chat.completion',
            'created': int(time.time()),
            'model': model,
            'choices': [
                {
                    'index': 0,
                    'message': {
                        'role': 'assistant',
                        'content': f"[Mock LLM Server response for model '{model}'] Recieved prompt: '{prompt}'"
                    },
                    'finish_reason': 'stop'
                }
            ],
            'usage': {
                'prompt_tokens': len(prompt.split()) + 10,
                'completion_tokens': 15,
                'total_tokens': len(prompt.split()) + 25
            }
        }
        
        self.send_response(200)
        self.send_header('Content-Type', 'application/json')
        self.end_headers()
        self.wfile.write(json.dumps(resp).encode('utf-8'))

PORT = 8088
server = socketserver.TCPServer(('127.0.0.1', PORT), MockLLMRequestHandler)
server.allow_reuse_address = True
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()
print(f'Mock LLM Server started locally on http://127.0.0.1:{PORT}')


## Part 2: Building raw POST Requests to the Mock LLM
Let's write a standard POST query to our local running endpoint.


In [ ]:
import requests

url = 'http://127.0.0.1:8088/v1/chat/completions'
headers = {
    'Content-Type': 'application/json',
    'Authorization': 'Bearer sk-mock-valid-key'
}
payload = {
    'model': 'gpt-4o',
    'messages': [
        {'role': 'user', 'content': 'What are vector embeddings?'}
    ],
    'temperature': 0.7
}

response = requests.post(url, headers=headers, json=payload)
print('Response Status:', response.status_code)
print('Response JSON:\n', json.dumps(response.json(), indent=2))


## Hands-On Exercise
**Task**: Write a python function `query_llm(prompt)` that calls the completions endpoint, extracts the content string response and the token details, prints them, and returns them.
Run this function with 3 different prompts, and save the list of results as a JSON array file named `data/first_api_calls.json`.


In [ ]:
# TODO: Complete the query_llm and prompt loop
def query_llm(prompt):
    url = 'http://127.0.0.1:8088/v1/chat/completions'
    headers = {
        'Content-Type': 'application/json',
        'Authorization': 'Bearer sk-mock-key'
    }
    payload = {
        'model': 'gpt-4o-mini',
        'messages': [{'role': 'user', 'content': prompt}]
    }
    r = requests.post(url, headers=headers, json=payload)
    data = r.json()
    
    content = data['choices'][0]['message']['content']
    usage = data['usage']
    
    return {
        'prompt': prompt,
        'response': content,
        'tokens_used': usage['total_tokens']
    }

prompts = [
    'Explain recursion in 1 line.',
    'What is a REST API?',
    'What is the capital of India?'
]

results = []
for p in prompts:
    res = query_llm(p)
    results.append(res)
    print(f"Prompt: '{p}' -> Response: '{res['response']}'")
    
with open('data/first_api_calls.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved completions history to data/first_api_calls.json')


In [ ]:
# Shutdown local server thread cleanly
server.shutdown()
server.server_close()
print('Local Mock server stopped.')
